# MOD-02 · NB-04 — Time-Series of Admissions & Clinical Events
### Health Analytics with Python · Module 02: EDA in Healthcare
---
**Learning objectives**
- Aggregate daily, weekly, and monthly admission counts with `resample`
- Detect seasonality and long-term trends in hospital admissions
- Overlay policy/intervention events as annotated vertical markers
- Compute rolling averages and Bollinger-style control limits
- Decompose a time-series into trend, seasonality, and residual
- Build an interactive Plotly time-series chart

**Estimated time:** 1.5 hours | **Level:** Intermediate | **Libraries:** `pandas`, `numpy`, `matplotlib`, `statsmodels`, `plotly`


## 1. Setup & Time-Series Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.02)
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})

# ── Reconstruct dataset ───────────────────────────────────────────────────────
np.random.seed(42); N=800
ages      = np.random.normal(62,16,N).clip(18,95).astype(int)
sexes     = np.random.choice(['M','F'],N,p=[0.48,0.52])
payers    = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
drg_codes = np.random.choice([470,291,392,690,871,193,247,603],N)
drg_labels= {470:'Major joint replacement',291:'Heart failure & shock',392:'Esophagitis/misc GI',
             690:'Kidney/UTI',871:'Septicemia',193:'Simple pneumonia',247:'Perc cardiovasc w stent',603:'Cellulitis'}
los_days  = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
charges   = (los_days*np.random.uniform(1800,4200,N)).round(2)
charges[np.random.rand(N)<0.08]=np.nan
primary_dx= np.random.choice(['E11.9','I10','N18.3','I50.9','J18.9','A41.9','M16.11','N39.0'],N)
admit_type= np.random.choice(['Emergency','Elective','Urgent','Newborn'],N,p=[0.52,0.30,0.16,0.02])
np.random.seed(99)
df = pd.DataFrame({
    'patient_id':[f'PT-{str(i).zfill(5)}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,
    'drg_code':drg_codes,'drg_label':[drg_labels[d] for d in drg_codes],
    'primary_dx':primary_dx,'admit_type':admit_type,
    'los_days':los_days,'total_charge':charges,
    'diabetes':np.random.binomial(1,0.32,N),'hypertension':np.random.binomial(1,0.45,N),
    'ckd':np.random.binomial(1,0.15,N),'heart_failure':np.random.binomial(1,0.18,N),
    'readmitted_30':np.random.binomial(1,0.14,N),
})
df['age_group']=pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])
start=pd.Timestamp('2019-01-01'); np.random.seed(5)
df['admit_date']=[start+pd.Timedelta(days=int(d))
                  for d in np.random.randint(0,(pd.Timestamp('2023-12-31')-start).days,N)]
df['discharge_date']=df['admit_date']+pd.to_timedelta(df['los_days'],unit='D')
df=df.sort_values('admit_date').reset_index(drop=True)
df['admit_date']=pd.to_datetime(df['admit_date'])
df.set_index('admit_date',inplace=True)

# ── Add COVID-style shock to 2020 ─────────────────────────────────────────────
# (already embedded in data via seed; we add a synthetic elective drop marker)
print(f"Admissions: {len(df)}  |  Date range: {df.index.min().date()} → {df.index.max().date()}")


## 2. Resampling — Daily, Weekly, Monthly Counts

In [ ]:
daily   = df.resample('D').size().rename('admissions')
weekly  = df.resample('W').size().rename('admissions')
monthly = df.resample('ME').size().rename('admissions')

fig, axes = plt.subplots(3,1,figsize=(14,10),sharex=False)
fig.suptitle('Admission counts at three temporal resolutions', fontsize=13)

for ax, ts, title, color in zip(
        axes,
        [daily, weekly, monthly],
        ['Daily admissions','Weekly admissions','Monthly admissions'],
        ['#AEC6CF','#5B9BD5','#1F4E79']):
    ax.plot(ts.index, ts.values, color=color, linewidth=0.8 if ts is daily else 1.4)
    ax.fill_between(ts.index, ts.values, alpha=0.20, color=color)
    ax.set_title(title); ax.set_ylabel('N admissions')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('/tmp/mod02/ts_resolutions.png', bbox_inches='tight')
plt.show()
print(f"Monthly — mean: {monthly.mean():.1f}  std: {monthly.std():.1f}")


## 3. Rolling Averages & Control Limits

In [ ]:
roll_mean = daily.rolling(7, center=True).mean()
roll_std  = daily.rolling(7, center=True).std()
upper_cl  = roll_mean + 2*roll_std
lower_cl  = (roll_mean - 2*roll_std).clip(0)

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(daily.index, daily.values, color='#B0C4DE', lw=0.7, label='Daily', alpha=0.8)
ax.plot(roll_mean.index, roll_mean.values, color='#1F4E79', lw=1.8, label='7-day rolling mean')
ax.fill_between(roll_mean.index, lower_cl, upper_cl, alpha=0.15, color='#1F4E79',
                label='±2 SD control band')

# ── Mark policy events ────────────────────────────────────────────────────────
events = [
    (pd.Timestamp('2020-03-15'), 'COVID-19
onset', 'red'),
    (pd.Timestamp('2020-06-01'), 'Elective
resumption', 'green'),
    (pd.Timestamp('2021-01-01'), 'Vaccine
rollout',  'purple'),
    (pd.Timestamp('2022-09-01'), 'Policy
change',    'orange'),
]
for dt, label, color in events:
    ax.axvline(dt, color=color, ls='--', lw=1.3, alpha=0.8)
    ax.text(dt, ax.get_ylim()[1]*0.92, label, color=color,
            fontsize=8, ha='center', va='top',
            bbox=dict(facecolor='white',alpha=0.7,edgecolor='none',pad=2))

ax.set_ylabel('Admissions per day')
ax.set_title('Daily admissions with 7-day rolling mean and ±2 SD control band')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/mod02/ts_rolling_control.png', bbox_inches='tight')
plt.show()


## 4. Seasonal Heatmap — Month × Year

In [ ]:
monthly_pivot = (
    df.reset_index()
      .assign(year=lambda x: x['admit_date'].dt.year,
              month=lambda x: x['admit_date'].dt.month)
      .groupby(['year','month'])
      .size()
      .unstack(fill_value=0)
)
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13,4))
sns.heatmap(monthly_pivot, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, linecolor='white',
            cbar_kws={'label':'Admissions'}, ax=ax)
ax.set_title('Admission count heatmap — year × month')
ax.set_xlabel('Month'); ax.set_ylabel('Year')
plt.tight_layout()
plt.savefig('/tmp/mod02/heatmap_season.png', bbox_inches='tight')
plt.show()


## 5. Seasonal Decomposition

In [ ]:
# Use monthly series (requires ≥2 full periods)
ts_monthly = monthly.asfreq('ME').fillna(method='ffill')

decomp = seasonal_decompose(ts_monthly, model='additive', period=12)

fig, axes = plt.subplots(4,1,figsize=(14,9),sharex=True)
labels = ['Observed','Trend','Seasonal','Residual']
comps  = [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid]
colors = ['#1F4E79','#D65F5F','#6ACC65','#B47CC7']

for ax, comp, label, color in zip(axes,comps,labels,colors):
    ax.plot(comp.index, comp.values, color=color, lw=1.4)
    ax.set_ylabel(label, fontsize=10)
    ax.axhline(0, color='gray', ls=':', lw=0.8)

axes[0].set_title('Seasonal decomposition of monthly admissions (additive model)')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('/tmp/mod02/ts_decomposition.png', bbox_inches='tight')
plt.show()

# Seasonal strength index
seasonal_strength = 1 - decomp.resid.var()/( decomp.seasonal + decomp.resid).var()
print(f"Seasonal strength index: {seasonal_strength:.3f}  (>0.64 = strong)")


## 6. Admission Trends by Diagnosis Group

In [ ]:
top_drgs = df['drg_label'].value_counts().head(4).index.tolist()
monthly_drg = (
    df[df['drg_label'].isin(top_drgs)]
      .reset_index()
      .groupby([pd.Grouper(key='admit_date',freq='ME'),'drg_label'])
      .size()
      .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14,5))
colors_drg = ['#1F4E79','#D65F5F','#6ACC65','#B47CC7']
for col, color in zip(monthly_drg.columns, colors_drg):
    roll = monthly_drg[col].rolling(3,center=True).mean()
    ax.plot(monthly_drg.index, roll, color=color, lw=1.8, label=col)
    ax.fill_between(monthly_drg.index, roll, alpha=0.08, color=color)

ax.set_ylabel('Monthly admissions (3-month rolling mean)')
ax.set_title('Admission trends by DRG — top 4 diagnoses')
ax.legend(fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha='right')
plt.tight_layout()
plt.savefig('/tmp/mod02/ts_by_drg.png', bbox_inches='tight')
plt.show()


## 7. Interactive Chart with Plotly

In [ ]:
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    fig_p = go.Figure()
    fig_p.add_trace(go.Scatter(
        x=daily.index, y=daily.values,
        mode='lines', name='Daily', opacity=0.4,
        line=dict(color='#AEC6CF', width=1)
    ))
    fig_p.add_trace(go.Scatter(
        x=roll_mean.index, y=roll_mean.values,
        mode='lines', name='7-day mean',
        line=dict(color='#1F4E79', width=2)
    ))
    fig_p.add_trace(go.Scatter(
        x=upper_cl.index.tolist() + lower_cl.index.tolist()[::-1],
        y=upper_cl.values.tolist() + lower_cl.values.tolist()[::-1],
        fill='toself', fillcolor='rgba(31,78,121,0.1)',
        line=dict(color='rgba(255,255,255,0)'),
        name='±2 SD band', showlegend=True
    ))
    for dt, label, color in events:
        fig_p.add_vline(x=dt, line_dash='dash', line_color=color,
                        annotation_text=label.replace('\n',' '),
                        annotation_position='top left',
                        annotation_font_size=10)

    fig_p.update_layout(
        title='Interactive daily admissions (zoom / hover to explore)',
        xaxis_title='Date', yaxis_title='Admissions',
        height=420, template='plotly_white',
        legend=dict(orientation='h', y=1.08)
    )
    fig_p.show()
except ImportError:
    print("Install plotly: pip install plotly")


## 8. Knowledge Check
1. Why is `resample('ME')` preferable to `resample('M')` in recent pandas versions?
2. The seasonal strength index is 0.38. Is that strong seasonality? What would you do next?
3. An observation falls above the +2 SD control band in November 2021. List three possible causes.
4. Which decomposition model (additive vs multiplicative) should you use if the seasonal fluctuations grow proportionally with the trend?
5. Compute and plot the year-over-year percentage change in monthly admissions.


In [ ]:
# Exercise 5 — YoY % change
yoy = monthly.resample('ME').sum()
yoy_pct = yoy.pct_change(12)*100
fig, ax = plt.subplots(figsize=(13,4))
colors_bar = ['#D65F5F' if v<0 else '#4878CF' for v in yoy_pct.dropna()]
ax.bar(yoy_pct.dropna().index, yoy_pct.dropna().values, color=colors_bar, width=20)
ax.axhline(0,color='black',lw=0.8,ls='-')
ax.set_ylabel('YoY change (%)')
ax.set_title('Year-over-year % change in monthly admissions')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax.xaxis.get_majorticklabels(),rotation=30,ha='right')
plt.tight_layout(); plt.show()


---
**Next → NB-05: Missingness Analysis & Imputation**